In [2]:
import numpy as np
import matplotlib.pyplot as plt

V_infinity = 1.0      # free stream velocity
alpha_deg = 0.0       # angle of attack in degrees
chord = 1.0           # chord length

panel_numbers = [20, 40, 80, 160, 320]   # number of panels for convergence

print("Free stream velocity:", V_infinity)
print("Angle of attack (deg):", alpha_deg)
print("Chord length:", chord)
print("Number of panels:", panel_numbers)

Free stream velocity: 1.0
Angle of attack (deg): 0.0
Chord length: 1.0
Number of panels: [20, 40, 80, 160, 320]


In [3]:
def circle_nodes(N, R):          # creating the circle geometry function

    x = np.zeros(N + 1)
    y = np.zeros(N + 1)

    for k in range(N + 1):

        phi = 2 * np.pi * k / N

        x[k] = R * np.cos(phi)
        y[k] = -R * np.sin(phi)

    return x, y


x_test, y_test = circle_nodes(8, 1.0)      #Now Testing the function again some random values

print("x nodes:", np.round(x_test, 4))
print("y nodes:", np.round(y_test, 4))

x nodes: [ 1.      0.7071  0.     -0.7071 -1.     -0.7071 -0.      0.7071  1.    ]
y nodes: [-0.     -0.7071 -1.     -0.7071 -0.      0.7071  1.      0.7071  0.    ]


In [6]:
def naca4_nodes(code, N, spacing, chord):      # created NACA 4-digit airfoil function

    if N % 2 != 0:
        raise ValueError("N must be even.")

    m = int(code[0]) / 100       
    p = int(code[1]) / 10        
    t = int(code[2:]) / 100      

    n = N // 2                   

    
    if spacing == "cosine":
        psi = np.linspace(0, np.pi, n + 1)
        x = 0.5 * (1 - np.cos(psi))
    elif spacing == "uniform":
        x = np.linspace(0, 1, n + 1)
    else:
        raise ValueError("spacing must be cosine or uniform")

    yt = np.zeros(n + 1)
    yc = np.zeros(n + 1)
    dyc = np.zeros(n + 1)

    for i in range(n + 1):

        
        yt[i] = 5 * t * (
            0.2969 * np.sqrt(x[i])
            - 0.1260 * x[i]
            - 0.3516 * x[i]**2
            + 0.2843 * x[i]**3
            - 0.1036 * x[i]**4
        )

        
        if m > 0:

            if x[i] < p:
                yc[i] = m / p**2 * (2 * p * x[i] - x[i]**2)
                dyc[i] = 2 * m / p**2 * (p - x[i])

            else:
                yc[i] = m / (1 - p)**2 * (1 - 2 * p + 2 * p * x[i] - x[i]**2)
                dyc[i] = 2 * m / (1 - p)**2 * (p - x[i])

    theta = np.arctan(dyc)

    # upper and lower surface
    x_upper = x - yt * np.sin(theta)
    y_upper = yc + yt * np.cos(theta)

    x_lower = x + yt * np.sin(theta)
    y_lower = yc - yt * np.cos(theta)

    
    x_nodes = np.concatenate([x_lower[::-1], x_upper[1:]]) * chord
    y_nodes = np.concatenate([y_lower[::-1], y_upper[1:]]) * chord

    return x_nodes, y_nodes


x_test, y_test = naca4_nodes("0012", 40, "cosine", chord)     # Testing for some random values

print("Number of nodes:", len(x_test))
print("First node (TE):", x_test[0], y_test[0])
print("Last node (TE) :", x_test[-1], y_test[-1])

Number of nodes: 41
First node (TE): 1.0 1.6653345369377347e-17
Last node (TE) : 1.0 -1.6653345369377347e-17
